# Paper 4 — 01 · Build + freeze contrastive sets

Build the five contrastive cells (EXPERIMENT_DESIGN §2): `harm_en`, `benign_en`, `harm_ro`, `benign_ro`, and the EN<->RO `parallel` set. Read sources from Paper 2 (RoSafetyBench) and HarmBench; label behavior with the Paper 2 `gpt-5-mini` judge for the execution probe. Freeze + SHA-256 the sets and the probe split — pre-registration (EXPERIMENT_DESIGN §11).

**Output:** `data/contrastive/<short>/*.jsonl`, `data/splits/probe_split.json`, both SHA-256'd.

In [10]:
%%capture
# Pinned to requirements.txt. Wheel-only on A100 / CUDA 12; restart rarely needed.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'datasets>=3.0' \
    'scikit-learn>=1.4' \
    'transformer-lens>=2.9' \
    'sae-lens>=4.0' \
    python-dotenv requests huggingface_hub ipywidgets pyyaml matplotlib seaborn -q


In [2]:
import os, json, gc, sys, hashlib, subprocess
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Secrets (Colab) -> env, so the Paper 2 judge + gated HF models work
#     end-to-end with no manual steps. Set these in Colab -> Secrets first. ---
try:
    from google.colab import userdata
    for _k in ("OPENROUTER_API_KEY", "HF_TOKEN"):
        try:
            _v = userdata.get(_k)
            if _v:
                os.environ[_k] = _v
        except Exception:
            print(f"[secrets] {_k} not set in Colab Secrets — add it if a cell needs it.")
except Exception:
    pass
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"], add_to_git_credential=False)

# --- Artifact root (persistent, on Drive) ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper4-interpretability")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")
PAPER3_ROOT = Path("/content/drive/MyDrive/PhD/paper3-alignment")

# --- Code root: use the repo synced on Drive if present, else clone the public
#     repo to /content. This makes the notebook self-provisioning — you do NOT
#     have to sync src/ + configs/ to Drive by hand. ---
REPO_URL = "https://github.com/robery567/rosafety-circuits.git"
if (DRIVE_ROOT / "src" / "paths.py").exists():
    CODE_ROOT = DRIVE_ROOT
else:
    CODE_ROOT = Path("/content/rosafety-circuits")
    if not (CODE_ROOT / "src" / "paths.py").exists():
        print("Paper 4 code not on Drive; cloning", REPO_URL)
        subprocess.run(["git", "clone", "-q", REPO_URL, str(CODE_ROOT)], check=True)
print("CODE_ROOT :", CODE_ROOT)
print("DRIVE_ROOT:", DRIVE_ROOT)

# --- data dirs (Drive, persistent across sessions) ---
DATA_DIR     = DRIVE_ROOT / "data"
CONTRAST_DIR = DATA_DIR / "contrastive"
ACT_DIR      = DATA_DIR / "activations"
PROBE_DIR    = DATA_DIR / "probes"
SPLITS_DIR   = DATA_DIR / "splits"
RESULTS_DIR  = DRIVE_ROOT / "results"
FIG_DIR      = DRIVE_ROOT / "figures"
LOGS_DIR     = DRIVE_ROOT / "logs"
for d in [CONTRAST_DIR, ACT_DIR, PROBE_DIR, SPLITS_DIR, RESULTS_DIR, FIG_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
CONFIG_DIR = CODE_ROOT / "configs"   # configs live in the repo, not in data/

# --- Reuse Paper 2 judge harness; Paper 4 src/ from CODE_ROOT ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))      # judges.py, llm_judge.py
sys.path.insert(0, str(CODE_ROOT / "src"))         # paths, capture, probes, patching, sae_utils, contrastive, behavioral

# --- A100 sanity ---
assert torch.cuda.is_available(), "Need a GPU runtime (A100 high-RAM)."
torch.backends.cuda.matmul.allow_tf32 = True
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)


Mounted at /content/drive


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Paper 4 code not on Drive; cloning https://github.com/robery567/rosafety-circuits.git
CODE_ROOT : /content/rosafety-circuits
DRIVE_ROOT: /content/drive/MyDrive/PhD/paper4-interpretability
GPU: NVIDIA A100-SXM4-40GB
torch: 2.7.1+cu126


## Configuration

In [3]:
# --- Anchor selection. Re-run the notebook once per anchor. ---
# SAE anchor (H1e available):  google/gemma-2-2b-it
# Cross-arch anchors:          Qwen/Qwen2.5-3B-Instruct, meta-llama/Llama-3.2-3B-Instruct
ANCHOR = "google/gemma-2-2b-it"

from paths import short_of, family_of
short  = short_of(ANCHOR)
family = family_of(ANCHOR)
print(f"ANCHOR : {ANCHOR}\nfamily : {family}\nshort  : {short}")


ANCHOR : google/gemma-2-2b-it
family : gemma2
short  : gemma-2-2b


## Sources

- `harm_en`: HarmBench standard + local core from crosslingual `text_en`.
- `benign_en`: XSTest-safe (matches RO over-refusal benign-but-risky semantics).
- `harm_ro` / `benign_ro`: RoSafetyBench (`paper2-benchmark/benchmark/expanded/`).
- `parallel`: RoSafetyBench crosslingual harmful pairs (`category=='harmful'`, patching-only).

In [4]:
import yaml
cfg = yaml.safe_load((CONFIG_DIR / 'experiments.yaml').read_text())
cells_cfg = cfg['contrastive_sets']['cells']
cells_cfg

{'harm_en': {'lang': 'en',
  'intent': 'harmful',
  'n': 250,
  'source': 'crosslingual_en core (36) + harmbench topup',
  'discriminator': 'category==harmful'},
 'benign_en': {'lang': 'en',
  'intent': 'benign',
  'n': 250,
  'source': 'xstest_safe (benign-but-risky, matches overrefusal)'},
 'harm_ro': {'lang': 'ro',
  'intent': 'harmful',
  'n': 100,
  'source': 'rosafetybench_toxicity+jailbreak (expected_behavior==refuse)'},
 'benign_ro': {'lang': 'ro',
  'intent': 'benign',
  'n': 100,
  'source': 'rosafetybench_overrefusal (expected_behavior==answer)'},
 'parallel': {'lang': 'en_ro',
  'intent': 'harmful',
  'n': 36,
  'source': 'rosafetybench_crosslingual category==harmful',
  'role': 'patching_only'}}

## 1. Build all cells (EN cells pull HarmBench + XSTest from HF)

`with_en=True` builds the EN cells too (needs `datasets` + network — fine on
Colab). RO cells + parallel + harm_en core are deterministic from the committed
Paper 2 files (already frozen locally; SHA-256 in `PREREGISTRATION.md`).

In [5]:
from contrastive import build_all, make_probe_split
expanded = PAPER2_ROOT / 'benchmark' / 'expanded'
out = CONTRAST_DIR / short
manifest = build_all(expanded, out, with_en=True)
import pprint; pprint.pprint(manifest['cells'])

README.md:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

standard/train-00000-of-00001.parquet:   0%|          | 0.00/12.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/3.53k [00:00<?, ?B/s]

data/gpt4-00000-of-00001.parquet:   0%|          | 0.00/133k [00:00<?, ?B/s]

...

Generating llama2orig split:   0%|          | 0/450 [00:00<?, ? examples/s]

Generating mistralguard split:   0%|          | 0/450 [00:00<?, ? examples/s]

Generating mistralinstruct split:   0%|          | 0/450 [00:00<?, ? examples/s]

Generating prompts split:   0%|          | 0/450 [00:00<?, ? examples/s]

{'benign_en': {'n': 250,
               'sha256': '5b181cf244b70e15e10630a246492e8d4e01d81013e74e9baf45982b2d65a174',
               'source': 'xstest_safe',
               'status': 'final'},
 'benign_ro': {'n': 100,
               'sha256': '1994b34a3fdb3b7c70f0cdb8e749b123e3dc9fb5df23f5a3747faf119389ced7',
               'source': 'rosafetybench_overrefusal',
               'status': 'final'},
 'harm_en': {'n': 236,
             'sha256': '8f4e3a8bb6daed55c46bbcdcde98072b34dfc9db72b7bcd3c079e52f9b88b3e9',
             'source': 'crosslingual_en+harmbench',
             'status': 'final'},
 'harm_ro': {'n': 100,
             'sha256': '1ca385525293ed23a6aa9e35982897433731ab73206fc5f556be44116e2611da',
             'source': 'rosafetybench_tox_jb',
             'status': 'final'},
 'parallel': {'n': 36,
              'sha256': '678428760dbf61f4d2220e8346e4c5059652f8af285cbb75c070f13e0aa23ae6',
              'source': 'rosafetybench_crosslingual',
              'status': 'final'}}


## 2. Load the anchor (for behavioral generation)

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(ANCHOR)
tok.padding_side = 'left'
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(ANCHOR, torch_dtype=torch.bfloat16, device_map='cuda').eval()
print('loaded', ANCHOR)

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

loaded google/gemma-2-2b-it


## 3. Behavioral labels (execution-probe target) — Paper 2 judge

Greedy one completion per prompt, judged refuse/comply by `gpt-5-mini`
(same protocol as Paper 2/3). Idempotent via the judge's on-disk cache.

In [7]:
from llm_judge import Judge          # Paper 2 src/
from behavioral import behavioral_labels_for_cells, gap_exhibiting_pairs
judge = Judge(model=cfg.get('judge', {}).get('primary', 'openai/gpt-5-mini')
              if isinstance(cfg.get('judge'), dict) else 'openai/gpt-5-mini')
labels_path = behavioral_labels_for_cells(model, tok, judge, out)
print('wrote', labels_path)
print(f'judge calls={judge.total_calls} cache_hits={judge.total_cache_hits}')

  [  20/236 ] 4.3/s  ETA 50s  cache 0  tokens 8790↑ 1010↓
  [  40/236 ] 4.7/s  ETA 42s  cache 0  tokens 17403↑ 2021↓
  [  60/236 ] 4.8/s  ETA 37s  cache 0  tokens 25369↑ 3030↓
  [  80/236 ] 4.9/s  ETA 32s  cache 0  tokens 34060↑ 4038↓
  [ 100/236 ] 4.6/s  ETA 29s  cache 0  tokens 42507↑ 5041↓
  [ 120/236 ] 4.7/s  ETA 25s  cache 0  tokens 51251↑ 6023↓
  [ 140/236 ] 4.6/s  ETA 21s  cache 0  tokens 60337↑ 7038↓
  [ 160/236 ] 4.7/s  ETA 16s  cache 0  tokens 69119↑ 8025↓
  [ 180/236 ] 4.8/s  ETA 12s  cache 0  tokens 78128↑ 9054↓
  [ 200/236 ] 4.8/s  ETA 8s  cache 0  tokens 86809↑ 10026↓
  [ 220/236 ] 4.9/s  ETA 3s  cache 0  tokens 95612↑ 11036↓
  [  20/250 ] 4.2/s  ETA 55s  cache 0  tokens 111404↑ 12903↓
  [  40/250 ] 4.2/s  ETA 50s  cache 0  tokens 120098↑ 13946↓
  [  60/250 ] 4.4/s  ETA 43s  cache 0  tokens 128824↑ 14963↓
  [  80/250 ] 4.4/s  ETA 38s  cache 0  tokens 137687↑ 15988↓
  [ 100/250 ] 4.4/s  ETA 34s  cache 0  tokens 146257↑ 17024↓
  [ 120/250 ] 4.5/s  ETA 29s  cache 0  tokens 1

## 4. Gap-exhibiting pairs (RO comply + EN refuse) → H1c patching set

In [8]:
pairs = gap_exhibiting_pairs(labels_path)
print(f'{len(pairs)} / {manifest["cells"]["parallel"]["n"]} parallel pairs exhibit the gap')
if len(pairs) < 15:
    print('WARNING: thin patching set — consider adding the bias subset (EXPERIMENT_LOG 2026-06-01).')

4 / 36 parallel pairs exhibit the gap


## 5. Freeze probe split + record SHA-256 (append to PREREGISTRATION.md §3)

In [9]:
split = make_probe_split(out)
print(f"train_en={len(split['train_en'])} eval_en={len(split['eval_en'])} eval_ro={len(split['eval_ro'])}")
print('probe_split sha256:', split['_sha256'])
print('harm_en sha256 :', manifest['cells']['harm_en']['sha256'])
print('benign_en sha256:', manifest['cells']['benign_en']['sha256'])
print('\n>> Append these three SHA-256s to PREREGISTRATION.md section 3 with today\'s date.')

train_en=340 eval_en=146 eval_ro=200
probe_split sha256: c99560b5e378c4ed796c005bad517cd58a58dae8fb95187fcb55a6e7d8472f50
harm_en sha256 : 8f4e3a8bb6daed55c46bbcdcde98072b34dfc9db72b7bcd3c079e52f9b88b3e9
benign_en sha256: 5b181cf244b70e15e10630a246492e8d4e01d81013e74e9baf45982b2d65a174

>> Append these three SHA-256s to PREREGISTRATION.md section 3 with today's date.
